In [ ]:
!pip install -q bedrock-agentcore strands-agents bedrock-agentcore-starter-toolkit boto3

In [ ]:
%%writefile hello_agent.py
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent

app = BedrockAgentCoreApp()
agent = Agent(system_prompt="You are a friendly assistant. Keep answers short.")

@app.entrypoint
def invoke(payload):
    prompt = payload.get("prompt", "Hello!")
    result = agent(prompt)
    return {"result": result.message}

if __name__ == "__main__":
    app.run()

In [ ]:
%%writefile requirements.txt
bedrock-agentcore
strands-agents
boto3

In [ ]:
import boto3, shutil, os, subprocess, sys, time, pathlib

region = boto3.Session().region_name or "us-east-1"
bucket = "bmo-332055852319-us-east-1-sams-test-bucket"
key = f"hello-agent/code-{int(time.time())}.zip"

shutil.rmtree("build", ignore_errors=True)
os.makedirs("build")

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "-r", "requirements.txt", "--target", "build",
    "--platform", "manylinux2014_aarch64", "--only-binary=:all:",
    "--python-version", "3.12", "--implementation", "cp",
    "--no-compile",                      # don't create .pyc files
], check=True)

shutil.copy("hello_agent.py", "build/")

# Remove any cache files that slipped in
for p in pathlib.Path("build").rglob("__pycache__"):
    shutil.rmtree(p, ignore_errors=True)
for p in pathlib.Path("build").rglob("*.py[co]"):
    p.unlink()

if os.path.exists("hello_agent.zip"):
    os.remove("hello_agent.zip")
shutil.make_archive("hello_agent", "zip", "build")

# Sanity check before uploading
import zipfile
names = zipfile.ZipFile("hello_agent.zip").namelist()
assert "hello_agent.py" in names, "entry point missing from zip root"
assert not any(n.endswith((".pyc", ".pyo")) or "__pycache__" in n for n in names), "cache files still present"
print(f"OK: {len(names)} files")

boto3.client("s3").upload_file("hello_agent.zip", bucket, key)
print(f"s3://{bucket}/{key}")